# LLM Inference Gateway: Colab benchmark

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. A full run takes about 60–90 minutes on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!git clone https://github.com/RishDevs/llm-inference-gateway.git
%cd llm-inference-gateway
!pip install -q -e ".[bench]"

In [ ]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 1. Gateway vs naive vs unbucketed, burst test, profiler

Writes JSON results to `results/`. Server logs are in `results/logs/`.

In [ ]:
!bash scripts/run_benchmarks.sh

## 2. vLLM reference

vLLM pins its own torch build, so it is installed after the gateway runs. `--dtype half` is required because the T4 has no bfloat16 support.

In [ ]:
!pip install -q vllm

In [ ]:
import subprocess, time, urllib.request
vllm = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--max-model-len", "2048", "--port", "8001"],
    stdout=open("results/logs/server-vllm.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(300):
    try:
        urllib.request.urlopen("http://localhost:8001/health", timeout=2)
        print("vLLM is up")
        break
    except Exception:
        time.sleep(2)

In [ ]:
!python -m bench.load_test --url http://localhost:8001 --model Qwen/Qwen2.5-1.5B-Instruct --label vllm --concurrency 1,8,16,32 --out results/vllm.json

In [ ]:
vllm.terminate(); vllm.wait()

## 3. Report

In [ ]:
!python -m bench.report results
from IPython.display import Markdown, display
display(Markdown(open("results/summary.md").read().replace("](", "](results/")))

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")